In [1]:
# -------- Two-picture hole detection + group center + clicks (OpenCV) --------
import cv2 as cv
import numpy as np
import time
import math

# ================= helpers =================
def _to_gray_norm(img):
    lab = cv.cvtColor(img, cv.COLOR_BGR2LAB)
    L, a, b = cv.split(lab)
    L = cv.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(L)
    bgr = cv.cvtColor(cv.merge([L, a, b]), cv.COLOR_LAB2BGR)
    gray = cv.cvtColor(bgr, cv.COLOR_BGR2GRAY)
    return cv.bilateralFilter(gray, 5, 50, 50)


def find_target_center(bgr, template=None):
    """Return center of the target aiming mark.

    template=None  ->  largest-dark-blob heuristic (works for any solid dark
                       shape: diamond, circle, square, filled cross, ...).
    template=<bgr> ->  normalised template matching (works for ANY shape,
                       including hollow rings, crosshairs, coloured marks).
                       Cropped automatically by set_clean_target().
    """
    if template is not None:
        t_gray = cv.cvtColor(template, cv.COLOR_BGR2GRAY) if template.ndim == 3 else template
        i_gray = cv.cvtColor(bgr, cv.COLOR_BGR2GRAY)
        res = cv.matchTemplate(i_gray, t_gray, cv.TM_CCOEFF_NORMED)
        _, _, _, max_loc = cv.minMaxLoc(res)
        th, tw = t_gray.shape[:2]
        return (int(max_loc[0] + tw // 2), int(max_loc[1] + th // 2))

    # --- fallback: dark-blob centroid (used once on clean target by set_clean_target) ---
    gray = cv.cvtColor(bgr, cv.COLOR_BGR2GRAY)
    inv = cv.GaussianBlur(cv.bitwise_not(gray), (5, 5), 0)
    _, th = cv.threshold(inv, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
    th = cv.morphologyEx(
        th,
        cv.MORPH_OPEN,
        cv.getStructuringElement(cv.MORPH_ELLIPSE, (5, 5)),
    )
    num, _, stats, cents = cv.connectedComponentsWithStats(th, 8)
    if num <= 1:
        h, w = gray.shape[:2]
        return (w // 2, h // 2)
    areas = stats[1:, cv.CC_STAT_AREA].astype(np.float64)
    # Weight-average ALL blobs that are at least 50% as large as the biggest one.
    # For a solid single-blob mark (diamond, dot) only one blob qualifies -- same
    # result as argmax.  For a disconnected mark (open crosshair) all four arms
    # qualify and their weighted centroid lands exactly on the geometric centre.
    keep = areas >= areas.max() * 0.5
    xs = cents[1:][keep, 0]
    ys = cents[1:][keep, 1]
    w  = areas[keep]
    cx = float(np.average(xs, weights=w))
    cy = float(np.average(ys, weights=w))
    return (int(round(cx)), int(round(cy)))


find_diamond_center = find_target_center  # backward-compat alias


def _register_to(ref_bgr, img_bgr, center_template=None):
    """
    Align shot to baseline using ONLY target center (pure translation).
    This guarantees correct vertical/horizontal alignment of the aiming mark
    even if the rest of the image (holes, cropping) differs.
    """
    cx_ref, cy_ref = find_target_center(ref_bgr, center_template)
    cx_img, cy_img = find_target_center(img_bgr, center_template)

    dx = cx_ref - cx_img
    dy = cy_ref - cy_img

    H = np.array(
        [
            [1, 0, dx],
            [0, 1, dy],
            [0, 0, 1],
        ],
        dtype=np.float32,
    )

    h, w = ref_bgr.shape[:2]
    warped = cv.warpPerspective(img_bgr, H, (w, h), borderValue=(255, 255, 255))
    return warped, H


def _contour_filter(bin_img, min_area=150, max_area=8000, circ_min=0.2):
    cnts, _ = cv.findContours(bin_img, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    out = []
    for c in cnts:
        area = cv.contourArea(c)
        if area < min_area or area > max_area:
            continue
        peri = cv.arcLength(c, True)
        if peri <= 0:
            continue
        circ = 4 * np.pi * area / (peri * peri)
        hull = cv.convexHull(c)
        solidity = area / (cv.contourArea(hull) + 1e-6)
        if circ >= circ_min and solidity > 0.7:
            M = cv.moments(c)
            if M["m00"] != 0:
                cx, cy = int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"])
                out.append((cx, cy, area, circ))
    return out


def estimate_group_center_from_diff_avg_robust(
    ref_gray,
    cur_gray,
    blur_ks=5,
    open_ks=3,
    close_ks=3,
    min_area=80,
    max_area=6000,
    circ_min=0.15,
    aim_xy=None,
    max_dist_px=None,
    use_area_weight=True,
):
    """
    Detect individual holes from abs-diff and return their (weighted) average center.
    Fallback: if no holes pass the filters (e.g., one big merged blob), return the
    intensity-weighted centroid of the dominant foreground component.
    Returns: (gcx, gcy, r_eq, union_mask, diff_img, holes)
    """
    diff = cv.absdiff(ref_gray, cur_gray)
    diff_blur = (
        cv.GaussianBlur(diff, (blur_ks, blur_ks), 0)
        if blur_ks and blur_ks >= 3
        else diff
    )

    # Threshold (Otsu; fallback to fixed if almost empty)
    _, th = cv.threshold(diff_blur, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
    if th.mean() < 1:  # nearly no FG; try fixed small threshold
        _, th = cv.threshold(diff_blur, 25, 255, cv.THRESH_BINARY)

    # Clean edges, but DON'T dilate (we want separate contours when possible)
    if open_ks and open_ks > 1:
        th = cv.morphologyEx(
            th,
            cv.MORPH_OPEN,
            cv.getStructuringElement(cv.MORPH_ELLIPSE, (open_ks, open_ks)),
        )
    if close_ks and close_ks > 1:
        th = cv.morphologyEx(
            th,
            cv.MORPH_CLOSE,
            cv.getStructuringElement(cv.MORPH_ELLIPSE, (close_ks, close_ks)),
        )

    cnts, _ = cv.findContours(th, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)

    holes = []
    union = np.zeros_like(th, np.uint8)

    ax, ay = aim_xy if aim_xy is not None else (None, None)
    max_d2 = None if max_dist_px is None else (max_dist_px * max_dist_px)

    # --- Try per-hole detection ---
    for c in cnts:
        area = cv.contourArea(c)
        if area < min_area or area > max_area:
            continue
        peri = cv.arcLength(c, True)
        if peri <= 0:
            continue
        circ = 4 * np.pi * area / (peri * peri)
        if circ < circ_min:
            continue
        M = cv.moments(c)
        if M["m00"] == 0:
            continue
        cx = M["m10"] / M["m00"]
        cy = M["m01"] / M["m00"]
        if ax is not None and max_d2 is not None:
            if (cx - ax) * (cx - ax) + (cy - ay) * (cy - ay) > max_d2:
                continue
        holes.append((cx, cy, area, circ))
        cv.drawContours(union, [c], -1, 255, -1)

    if holes:
        w = (
            np.array([h[2] for h in holes], np.float64)
            if use_area_weight
            else np.ones(len(holes), np.float64)
        )
        xs = np.array([h[0] for h in holes], np.float64)
        ys = np.array([h[1] for h in holes], np.float64)
        gcx = float((xs * w).sum() / w.sum())
        gcy = float((ys * w).sum() / w.sum())
        total_area = float(np.sum(union > 0))
        if total_area <= 0:
            total_area = float(np.sum([h[2] for h in holes]))
        r_eq = float(np.sqrt(max(total_area, 1.0) / np.pi))
        return gcx, gcy, r_eq, union, diff, holes

    # --- Fallback: single blob centroid (merged hits) ---
    num, labels, stats, cents = cv.connectedComponentsWithStats(
        th, connectivity=8
    )
    if num <= 1:
        return None, None, None, th, diff, []  # truly nothing

    cand_ids = list(range(1, num))
    if aim_xy is not None:
        # pick component closest to aim
        ax, ay = aim_xy
        d2, idx = min(
            (
                (cents[i][0] - ax) ** 2 + (cents[i][1] - ay) ** 2,
                i,
            )
            for i in cand_ids
        )
    else:
        # largest area
        idx = 1 + int(np.argmax(stats[1:, cv.CC_STAT_AREA]))

    mask = (labels == idx).astype(np.uint8)
    m = cv.moments(diff.astype(np.float32) * mask, binaryImage=False)
    if m["m00"] <= 1e-6:
        # fall back to geometric centroid
        cx, cy = cents[idx]
    else:
        cx, cy = (m["m10"] / m["m00"], m["m01"] / m["m00"])

    area = float(stats[idx, cv.CC_STAT_AREA])
    r_eq = float(np.sqrt(max(area, 1.0) / np.pi))
    return (
        float(cx),
        float(cy),
        float(r_eq),
        (mask * 255).astype(np.uint8),
        diff,
        [],
    )


def _format_clicks(wind, elev):
    dir_x = "LEFT"  if wind > 0 else "RIGHT"
    dir_y = "UP"    if elev > 0 else "DOWN"
    return dir_x, dir_y, abs(wind), abs(elev)


def _overlay_click_banner(img, wind, elev):
    """Large, centered banner at the top with clicks."""
    dir_x, dir_y, wx, ey = _format_clicks(wind, elev)
    text = f"{wx:.1f} clicks {dir_x}   |   {ey:.1f} clicks {dir_y}"

    h, w = img.shape[:2]
    pad_y = 12
    bar_h = 54
    overlay = img.copy()
    cv.rectangle(overlay, (0, 0), (w, bar_h + 2 * pad_y), (0, 0, 0), -1)
    cv.addWeighted(overlay, 0.55, img, 0.45, 0, img)
    cv.putText(
        img, text, (20, bar_h),
        cv.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2, cv.LINE_AA,
    )


def _draw_aim_to_group_arrow(img, aim_xy, group_xy, color=(0, 255, 255)):
    if aim_xy is None or group_xy is None:
        return
    cv.arrowedLine(img,
                   (int(aim_xy[0]), int(aim_xy[1])),
                   (int(group_xy[0]), int(group_xy[1])),
                   color, 2, cv.LINE_AA, tipLength=0.03)


def _resize_to_box(img, max_w=1500, max_h=1000):
    H, W = img.shape[:2]
    s = min(max_w / max(W, 1), max_h / max(H, 1))
    if s >= 1.0:
        return img.copy()
    return cv.resize(img, (int(W * s), int(H * s)), interpolation=cv.INTER_AREA)


def show_operator_view(
    base_img, wind, elev,
    win_name="Zeroing - Operator View",
    fullscreen=False, save_key="s",
    max_w=1500, max_h=1000,
):
    img = base_img.copy()
    _overlay_click_banner(img, wind, elev)
    disp = _resize_to_box(img, max_w=max_w, max_h=max_h)

    cv.namedWindow(win_name, cv.WINDOW_NORMAL)
    if fullscreen:
        cv.setWindowProperty(win_name, cv.WND_PROP_FULLSCREEN, cv.WINDOW_FULLSCREEN)

    cv.imshow(win_name, disp)
    while True:
        k = cv.waitKey(0) & 0xFF
        if k in (ord("q"), 27):
            break
        if k == ord(save_key):
            fn = f"zeroing_{int(time.time())}.png"
            cv.imwrite(fn, disp)
            print(f"[saved] {fn}")
    cv.destroyWindow(win_name)


def pixels_to_clicks(dx_px, dy_px, px_per_cm, distance_m, click_value_cm=0.5):
    """
    Convert dx, dy offsets (pixels) -> correction in clicks.

    dx_px, dy_px: group_center - aim_center  (image coords)
      +dx = group to the RIGHT of aim -> wind > 0 -> LEFT correction
      +dy = group BELOW aim           -> elev > 0 -> UP correction
    """
    dx_cm = dx_px / px_per_cm
    dy_cm = dy_px / px_per_cm
    wind = dx_cm / click_value_cm
    elev = dy_cm / click_value_cm
    return wind, elev


def _overlay_click_box(img, wind, elev, pos=(18, 28)):
    """Draw a semi-transparent box with click instructions."""
    dir_x, dir_y, wx, ey = _format_clicks(wind, elev)
    lines = [
        f"Windage: {wx:.1f} clicks {dir_x}",
        f"Elevation: {ey:.1f} clicks {dir_y}",
    ]
    pad, lh = 8, 22
    w = max(cv.getTextSize(s, cv.FONT_HERSHEY_SIMPLEX, 0.6, 1)[0][0] for s in lines) + 2 * pad
    h = len(lines) * lh + 2 * pad
    x0, y0 = pos
    x1, y1 = x0 + w, y0 + h
    overlay = img.copy()
    cv.rectangle(overlay, (x0, y0), (x1, y1), (20, 20, 20), -1)
    cv.addWeighted(overlay, 0.55, img, 0.45, 0, img)
    y = y0 + pad + 16
    for s in lines:
        cv.putText(img, s, (x0 + pad, y), cv.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv.LINE_AA)
        y += lh


def _draw_correction_arrow(img, from_xy, to_xy, color=(0, 200, 0)):
    if from_xy is None or to_xy is None:
        return
    cv.arrowedLine(img,
                   (int(from_xy[0]), int(from_xy[1])),
                   (int(to_xy[0]), int(to_xy[1])),
                   color, 3, cv.LINE_AA, tipLength=0.035)


def estimate_px_per_cm_from_grid(img, grid_size_cm=1.0):
    """Estimate pixels-per-cm from the target grid lines."""
    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY) if img.ndim == 3 else img
    g = cv.GaussianBlur(gray, (3, 3), 0)
    edges = cv.Canny(g, 60, 120)
    edges = cv.dilate(edges, cv.getStructuringElement(cv.MORPH_RECT, (3, 3)), 1)
    px_x = _period_from_profile(edges.sum(axis=0))
    px_y = _period_from_profile(edges.sum(axis=1))
    vals = [v for v in (px_x, px_y) if v is not None]
    return (float(np.mean(vals)) / grid_size_cm) if vals else 40.0


def _period_from_profile(p):
    p = p.astype(np.float32)
    p = (p - p.mean()) / (p.std() + 1e-6)
    ac = np.correlate(p, p, mode="full")[len(p) - 1:]
    min_lag = 6
    if min_lag >= len(ac):
        return None
    return min_lag + int(np.argmax(ac[min_lag:]))


# ---- debug panel helpers ----
def _to_bgr(img):
    return cv.cvtColor(img, cv.COLOR_GRAY2BGR) if img.ndim == 2 else img


def _titled(img, title):
    out = img.copy()
    cv.rectangle(out, (0, 0), (out.shape[1], 24), (30, 30, 30), -1)
    cv.putText(out, title, (8, 18), cv.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1, cv.LINE_AA)
    return out


def _fit_into(img, target_wh):
    w, h = target_wh
    H, W = img.shape[:2]
    s = min(w / max(W, 1), h / max(H, 1))
    im = cv.resize(img, (max(1, int(W * s)), max(1, int(H * s))), cv.INTER_AREA)
    canvas = np.full((h, w, 3), 45, np.uint8)
    y0 = (h - im.shape[0]) // 2
    x0 = (w - im.shape[1]) // 2
    canvas[y0:y0 + im.shape[0], x0:x0 + im.shape[1]] = im
    return canvas


def _panel(named, cell=(420, 420), cols=4, gap=8, bg=(18, 18, 18)):
    w, h = cell
    tiles = []
    for t, im in named:
        b = np.full((h, w, 3), 45, np.uint8) if im is None else _fit_into(_to_bgr(im), (w, h))
        tiles.append(_titled(b, t))
    r = (len(tiles) + cols - 1) // cols
    th, tw = tiles[0].shape[:2]
    panel = np.full(((th + gap) * r + gap, (tw + gap) * cols + gap, 3), bg, np.uint8)
    i = 0
    for R in range(r):
        for C in range(cols):
            if i >= len(tiles):
                break
            y = gap + R * (th + gap)
            x = gap + C * (tw + gap)
            panel[y:y + th, x:x + tw] = tiles[i]
            i += 1
    return panel


def _draw_holes_scaled(bgr, holes, color=(0, 255, 0)):
    vis = bgr.copy()
    for (x, y, area, _) in holes:
        r = max(4, int(np.sqrt(area / np.pi)))
        cv.circle(vis, (x, y), r, color, 2)
        cv.circle(vis, (x, y), max(2, r // 3), color, -1)
    return vis


# ================= ZeroingSession =================

class ZeroingSession:
    """
    Session object that keeps a moving baseline.

    Usage:
        sess = ZeroingSession(distance_m=50.0, grid_cm=1.0, click_value_cm=0.5)
        sess.set_clean_target(cv.imread("clean.png"))
        res1 = sess.process_shot(cv.imread("shot1.png"))
        res2 = sess.process_shot(cv.imread("shot2.png"))
    """

    def __init__(self, distance_m=50.0, grid_cm=1.0, click_value_cm=0.5):
        self.distance_m    = distance_m
        self.grid_cm       = grid_cm
        self.click_value_cm = click_value_cm

        self.baseline_bgr  = None
        self.baseline_gray = None
        self.center_template = None
        self.round_index   = 0

    def set_clean_target(self, img_bgr):
        if img_bgr is None:
            raise ValueError("set_clean_target: img_bgr is None")
        self.baseline_bgr  = img_bgr.copy()
        self.baseline_gray = _to_gray_norm(self.baseline_bgr)
        self.round_index   = 0

        cx, cy = find_target_center(img_bgr)
        h, w   = img_bgr.shape[:2]
        half   = max(30, int(min(h, w) * 0.08))
        x1, y1 = max(0, cx - half), max(0, cy - half)
        x2, y2 = min(w, cx + half), min(h, cy + half)
        self.center_template = img_bgr[y1:y2, x1:x2].copy()
        print(f"[session] baseline set - center template "
              f"{self.center_template.shape[1]}x{self.center_template.shape[0]}px "
              f"at ({cx},{cy})")

    def process_shot(self, shot_bgr, show_debug=True, show_operator=True, win_prefix="Zeroing"):
        if self.baseline_bgr is None:
            raise RuntimeError("Baseline not set. Call set_clean_target() first.")
        if shot_bgr is None:
            raise ValueError("process_shot: shot_bgr is None")

        self.round_index += 1
        print(f"\n[session] processing round #{self.round_index}")

        shot_warped, _ = _register_to(self.baseline_bgr, shot_bgr, self.center_template)
        shot_gray      = _to_gray_norm(shot_warped)
        aim_px         = find_target_center(shot_warped, self.center_template)

        gcx, gcy, r_eq, group_mask, diff_s, holes = \
            estimate_group_center_from_diff_avg_robust(
                self.baseline_gray, shot_gray,
                blur_ks=5, open_ks=3, close_ks=3,
                min_area=80, max_area=6000, circ_min=0.15,
                aim_xy=aim_px, max_dist_px=None, use_area_weight=True,
            )

        px_per_cm = estimate_px_per_cm_from_grid(shot_warped, self.grid_cm)

        if gcx is not None:
            dx, dy = gcx - aim_px[0], gcy - aim_px[1]
            wind, elev = pixels_to_clicks(dx, dy, px_per_cm, self.distance_m,
                                          click_value_cm=self.click_value_cm)
            print(f"[session] group center=({gcx:.1f},{gcy:.1f}); "
                  f"windage {wind:+.1f} clicks, elevation {elev:+.1f} clicks")
        else:
            print("[session] group center not found.")
            wind = elev = 0.0

        # --- overlays ---
        overlay = shot_warped.copy()
        if gcx is not None:
            cv.circle(overlay, (int(gcx), int(gcy)), max(6, int(r_eq)), (0, 255, 255), 2)
        cv.drawMarker(overlay, aim_px, (255, 255, 0), cv.MARKER_CROSS, 18, 2)
        if gcx is not None:
            _draw_aim_to_group_arrow(overlay, aim_px, (gcx, gcy))
            _overlay_click_box(overlay, wind, elev, pos=(18, 28))

        op = shot_warped.copy()
        if gcx is not None:
            cv.circle(op, (int(gcx), int(gcy)), max(6, int(r_eq)), (0, 255, 255), 2)
        cv.drawMarker(op, aim_px, (255, 255, 0), cv.MARKER_CROSS, 18, 2)
        if gcx is not None:
            _draw_correction_arrow(op, (gcx, gcy), aim_px, color=(0, 200, 0))

        if show_operator:
            show_operator_view(op, wind, elev,
                               win_name=f"{win_prefix} - Operator View",
                               max_w=1500, max_h=1000)

        if show_debug:
            diff   = cv.absdiff(self.baseline_gray, shot_gray)
            inv    = cv.bitwise_not(shot_gray)
            tophat = cv.morphologyEx(inv, cv.MORPH_TOPHAT,
                                     cv.getStructuringElement(cv.MORPH_ELLIPSE, (9, 9)))
            mix    = cv.addWeighted(diff, 0.6, tophat, 0.4, 0)
            th     = cv.adaptiveThreshold(mix, 255, cv.ADAPTIVE_THRESH_GAUSSIAN_C,
                                          cv.THRESH_BINARY, 31, 2)
            th     = cv.medianBlur(th, 5)
            th     = cv.morphologyEx(th, cv.MORPH_OPEN,
                                     cv.getStructuringElement(cv.MORPH_ELLIPSE, (5, 5)))
            th     = cv.morphologyEx(th, cv.MORPH_CLOSE,
                                     cv.getStructuringElement(cv.MORPH_ELLIPSE, (7, 7)))
            panel  = _panel(
                [("Warped + Group/Aim", overlay), ("Baseline Gray", self.baseline_gray),
                 ("Shot Gray", shot_gray), ("Abs Diff", diff),
                 ("Top-hat (inv L)", tophat), ("Mix (Diff+Tophat)", mix),
                 ("Binary Mask", th), ("Group Mask", group_mask)],
                cell=(420, 420), cols=4, gap=8,
            )
            cv.imshow(f"{win_prefix} - Debug Panel (round {self.round_index})", panel)
            cv.waitKey(0)
            cv.destroyWindow(f"{win_prefix} - Debug Panel (round {self.round_index})")

        self.baseline_bgr  = shot_warped.copy()
        self.baseline_gray = shot_gray.copy()

        return {
            "gc":          (float(gcx), float(gcy)) if gcx is not None else None,
            "aim":         (float(aim_px[0]), float(aim_px[1])),
            "wind":        float(wind),
            "elev":        float(elev),
            "shot_warped": shot_warped,
            "overlay":     overlay,
            "operator":    op,
        }


# ================= UI helpers =================

# Colour palette (Catppuccin Mocha-inspired dark theme)
_C = {
    "bg":      "#1e1e2e",
    "surface": "#313244",
    "border":  "#45475a",
    "fg":      "#cdd6f4",
    "sub":     "#6c7086",
    "accent":  "#89b4fa",   # blue
    "green":   "#a6e3a1",
    "red":     "#f38ba8",
    "btn":     "#45475a",
    "btn_hov": "#585b70",
}

IMG_TYPES = [
    ("Images", "*.png *.jpg *.jpeg *.bmp *.tiff *.tif"),
    ("All files", "*.*"),
]


def _apply_style(root):
    """Apply dark ttk theme to a Tk root window."""
    import tkinter.ttk as ttk
    s = ttk.Style(root)
    s.theme_use("clam")
    s.configure(".",
                 background=_C["bg"], foreground=_C["fg"],
                 font=("Segoe UI", 10), borderwidth=0)
    s.configure("TFrame",      background=_C["bg"])
    s.configure("TLabel",      background=_C["bg"], foreground=_C["fg"])
    s.configure("Sub.TLabel",  background=_C["bg"], foreground=_C["sub"],
                font=("Segoe UI", 8))
    s.configure("Title.TLabel", background=_C["bg"], foreground=_C["accent"],
                font=("Segoe UI", 15, "bold"))
    s.configure("TEntry",
                fieldbackground=_C["surface"], foreground=_C["fg"],
                insertcolor=_C["fg"], relief="flat", padding=4)
    s.configure("TButton",
                background=_C["btn"], foreground=_C["fg"],
                relief="flat", padding=(10, 6))
    s.map("TButton",
          background=[("active", _C["btn_hov"]), ("disabled", _C["surface"])],
          foreground=[("disabled", _C["sub"])])
    s.configure("Accent.TButton",
                background=_C["accent"], foreground=_C["bg"],
                font=("Segoe UI", 10, "bold"), padding=(10, 8))
    s.map("Accent.TButton",
          background=[("active", "#74c7ec"), ("disabled", _C["surface"])],
          foreground=[("disabled", _C["sub"])])
    s.configure("Danger.TButton",
                background=_C["red"], foreground=_C["bg"],
                font=("Segoe UI", 10, "bold"), padding=(10, 8))
    s.map("Danger.TButton",
          background=[("active", "#eba0ac")])
    s.configure("Warning.TButton",
                background="#f9e2af", foreground=_C["bg"],
                font=("Segoe UI", 10, "bold"), padding=(10, 8))
    s.map("Warning.TButton",
          background=[("active", "#f5c842"), ("disabled", _C["surface"])],
          foreground=[("disabled", _C["sub"])])
    return s


def _center_window(win, w, h):
    win.update_idletasks()
    sw = win.winfo_screenwidth()
    sh = win.winfo_screenheight()
    win.geometry(f"{w}x{h}+{(sw - w) // 2}+{(sh - h) // 2}")


def _divider(parent):
    """Thin horizontal separator."""
    import tkinter as tk
    return tk.Frame(parent, bg=_C["border"], height=1)


# ================= Setup dialog =================

def _run_setup_dialog():
    """
    Show a single dark-themed setup window.
    Returns dict(distance_m, clicks_per_cm, clean_path) or None if cancelled.
    """
    import tkinter as tk
    import tkinter.ttk as ttk
    from tkinter import filedialog, messagebox

    root = tk.Tk()
    root.title("Zeroing Calculator")
    root.configure(bg=_C["bg"])
    root.resizable(False, False)
    root.attributes("-topmost", True)
    _apply_style(root)

    fr = ttk.Frame(root, padding=(28, 22, 28, 26))
    fr.pack(fill="both", expand=True)

    ttk.Label(fr, text="Zeroing Calculator", style="Title.TLabel").grid(
        row=0, column=0, columnspan=3, sticky="w", pady=(0, 20))

    # ── Distance ─────────────────────────────────────────────────────
    ttk.Label(fr, text="Distance to target").grid(row=1, column=0, sticky="w", pady=5)
    dist_var = tk.StringVar(value="50")
    dist_entry = ttk.Entry(fr, textvariable=dist_var, width=8)
    dist_entry.grid(row=1, column=1, sticky="w", padx=(10, 4))
    ttk.Label(fr, text="m").grid(row=1, column=2, sticky="w")

    # ── Clicks / cm ──────────────────────────────────────────────────
    ttk.Label(fr, text="Scope clicks / cm").grid(row=2, column=0, sticky="w", pady=5)
    clicks_var = tk.StringVar(value="2")
    clicks_entry = ttk.Entry(fr, textvariable=clicks_var, width=8)
    clicks_entry.grid(row=2, column=1, sticky="w", padx=(10, 4))
    hint_var = tk.StringVar(value="= 0.50 cm / click")
    ttk.Label(fr, textvariable=hint_var, style="Sub.TLabel").grid(
        row=2, column=2, sticky="w")

    def _update_hint(*_):
        try:
            c = float(clicks_var.get())
            hint_var.set(f"= {1/c:.2f} cm / click" if c > 0 else "")
        except ValueError:
            hint_var.set("")

    clicks_var.trace_add("write", _update_hint)

    # ── Divider ──────────────────────────────────────────────────────
    _divider(fr).grid(row=3, column=0, columnspan=3, sticky="ew", pady=18)

    # ── Clean target ─────────────────────────────────────────────────
    ttk.Label(fr, text="Clean target image").grid(row=4, column=0, sticky="w", pady=(0, 4))
    browse_btn = ttk.Button(fr, text="Browse...")
    browse_btn.grid(row=4, column=2, sticky="e")

    clean_name_var = tk.StringVar(value="No file selected")
    ttk.Label(fr, textvariable=clean_name_var, style="Sub.TLabel",
              wraplength=260, justify="left").grid(
        row=5, column=0, columnspan=3, sticky="w")

    clean_path_box = [None]

    def browse():
        p = filedialog.askopenfilename(
            title="Select CLEAN target image (no bullet holes)",
            filetypes=IMG_TYPES,
        )
        if p:
            clean_path_box[0] = p
            clean_name_var.set(p.replace("\\", "/").split("/")[-1])
            start_btn.state(["!disabled"])

    browse_btn.configure(command=browse)

    # ── Divider ──────────────────────────────────────────────────────
    _divider(fr).grid(row=6, column=0, columnspan=3, sticky="ew", pady=18)

    # ── Start button ─────────────────────────────────────────────────
    result = {}

    def on_start():
        try:
            d = float(dist_var.get())
            c = float(clicks_var.get())
            if d <= 0 or c <= 0:
                raise ValueError
        except ValueError:
            messagebox.showerror("Invalid input",
                                 "Distance and clicks/cm must be positive numbers.",
                                 parent=root)
            return
        result["distance_m"]    = d
        result["clicks_per_cm"] = c
        result["clean_path"]    = clean_path_box[0]
        root.quit()

    start_btn = ttk.Button(fr, text="Start Session",
                           style="Accent.TButton", command=on_start,
                           state="disabled")
    start_btn.grid(row=7, column=0, columnspan=3, sticky="ew")

    _center_window(root, 370, 330)
    root.mainloop()
    root.destroy()
    return result if result else None


# ================= Session control panel =================

def _run_session_panel(sess, distance_m, clicks_per_cm):
    """
    Persistent session control window.
    Shows round counter + last result; drives per-round file picking.
    """
    import tkinter as tk
    import tkinter.ttk as ttk
    from tkinter import filedialog, messagebox

    root = tk.Tk()
    root.title("Zeroing Session")
    root.configure(bg=_C["bg"])
    root.resizable(False, False)
    root.attributes("-topmost", True)
    _apply_style(root)

    fr = ttk.Frame(root, padding=(28, 22, 28, 26))
    fr.pack(fill="both", expand=True)

    # ── Header ───────────────────────────────────────────────────────
    ttk.Label(fr, text="Zeroing Session", style="Title.TLabel").pack(anchor="w")
    ttk.Label(fr,
              text=f"  {distance_m:.0f} m   |   {clicks_per_cm:.2g} clicks / cm",
              style="Sub.TLabel").pack(anchor="w", pady=(2, 16))

    # ── Round status ─────────────────────────────────────────────────
    round_var = tk.StringVar(value="Load the first shot image to begin.")
    ttk.Label(fr, textvariable=round_var, style="Sub.TLabel",
              font=("Segoe UI", 9, "italic")).pack(anchor="w")

    # ── Result card ──────────────────────────────────────────────────
    card = tk.Frame(fr, bg=_C["surface"], padx=20, pady=14)
    card.pack(fill="x", pady=(10, 0))

    # Windage column
    tk.Label(card, text="WINDAGE", bg=_C["surface"], fg=_C["sub"],
             font=("Segoe UI", 8, "bold")).grid(row=0, column=0, sticky="w")
    wind_val  = tk.StringVar(value="—")
    wind_dir  = tk.StringVar(value="")
    tk.Label(card, textvariable=wind_val, bg=_C["surface"], fg=_C["fg"],
             font=("Segoe UI", 28, "bold")).grid(row=1, column=0, sticky="w")
    tk.Label(card, textvariable=wind_dir, bg=_C["surface"], fg=_C["accent"],
             font=("Segoe UI", 11, "bold")).grid(row=2, column=0, sticky="w")

    # Vertical divider
    tk.Frame(card, bg=_C["border"], width=1).grid(
        row=0, column=1, rowspan=3, padx=24, sticky="ns")

    # Elevation column
    tk.Label(card, text="ELEVATION", bg=_C["surface"], fg=_C["sub"],
             font=("Segoe UI", 8, "bold")).grid(row=0, column=2, sticky="w")
    elev_val  = tk.StringVar(value="—")
    elev_dir  = tk.StringVar(value="")
    tk.Label(card, textvariable=elev_val, bg=_C["surface"], fg=_C["fg"],
             font=("Segoe UI", 28, "bold")).grid(row=1, column=2, sticky="w")
    tk.Label(card, textvariable=elev_dir, bg=_C["surface"], fg=_C["accent"],
             font=("Segoe UI", 11, "bold")).grid(row=2, column=2, sticky="w")

    # ── Buttons ──────────────────────────────────────────────────────
    _divider(fr).pack(fill="x", pady=20)

    round_num = [0]

    def load_shot():
        round_num[0] += 1
        shot_path = filedialog.askopenfilename(
            title=f"Round #{round_num[0]}: select shot image",
            filetypes=IMG_TYPES,
        )
        if not shot_path:
            round_num[0] -= 1
            return

        shot = cv.imread(shot_path)
        if shot is None:
            messagebox.showerror("Error", f"Failed to load:\n{shot_path}", parent=root)
            round_num[0] -= 1
            return

        round_var.set(f"Processing round #{round_num[0]} ...")
        load_btn.state(["disabled"])
        reset_btn.state(["disabled"])
        end_btn.state(["disabled"])
        wind_val.set("—")
        elev_val.set("—")
        wind_dir.set("")
        elev_dir.set("")
        root.update()

        res = sess.process_shot(shot, show_debug=True, show_operator=True,
                                win_prefix=f"Round {round_num[0]}")

        w, e = res["wind"], res["elev"]
        dw = "clicks LEFT"  if w > 0 else "clicks RIGHT"
        de = "clicks UP"    if e > 0 else "clicks DOWN"
        wind_val.set(f"{abs(w):.1f}")
        wind_dir.set(dw)
        elev_val.set(f"{abs(e):.1f}")
        elev_dir.set(de)
        round_var.set(f"Round #{round_num[0]} complete  —  ready for next shot")

        load_btn.state(["!disabled"])
        reset_btn.state(["!disabled"])
        end_btn.state(["!disabled"])

    def end_session():
        root.quit()

    def reset_session():
        new_path = filedialog.askopenfilename(
            title="Select NEW clean target image to reset session",
            filetypes=IMG_TYPES,
        )
        if not new_path:
            return
        new_clean = cv.imread(new_path)
        if new_clean is None:
            messagebox.showerror("Error", f"Failed to load:\n{new_path}", parent=root)
            return
        load_btn.state(["disabled"])
        reset_btn.state(["disabled"])
        end_btn.state(["disabled"])
        round_var.set("Resetting session ...")
        root.update()
        sess.set_clean_target(new_clean)
        round_num[0] = 0
        wind_val.set("—")
        elev_val.set("—")
        wind_dir.set("")
        elev_dir.set("")
        round_var.set("Session reset — load the first shot image to begin.")
        load_btn.state(["!disabled"])
        reset_btn.state(["!disabled"])
        end_btn.state(["!disabled"])

    load_btn = ttk.Button(fr, text="Load Shot Image",
                          style="Accent.TButton", command=load_shot)
    load_btn.pack(fill="x", pady=(0, 8))

    btn_row = ttk.Frame(fr)
    btn_row.pack(fill="x")

    reset_btn = ttk.Button(btn_row, text="Reset Session",
                           style="Warning.TButton", command=reset_session)
    reset_btn.pack(side="left", fill="x", expand=True, padx=(0, 8))

    end_btn = ttk.Button(btn_row, text="End Session",
                         style="Danger.TButton", command=end_session)
    end_btn.pack(side="left", fill="x", expand=True)

    _center_window(root, 420, 400)
    root.mainloop()
    root.destroy()
    print(f"[session] ended after {round_num[0]} round(s).")


# ================= Entry point =================

def run_interactive_session():
    """Launch setup dialog then session control panel."""
    params = _run_setup_dialog()
    if params is None:
        print("[session] cancelled.")
        return

    distance_m    = params["distance_m"]
    clicks_per_cm = params["clicks_per_cm"]
    click_value_cm = 1.0 / max(clicks_per_cm, 0.01)
    clean_path    = params["clean_path"]

    print(f"[session] distance={distance_m:.0f} m | "
          f"{clicks_per_cm:.2g} clicks/cm -> {click_value_cm:.4f} cm/click")

    clean = cv.imread(clean_path)
    if clean is None:
        print(f"[session] failed to load: {clean_path}")
        return

    sess = ZeroingSession(distance_m=distance_m, grid_cm=1.0,
                          click_value_cm=click_value_cm)
    sess.set_clean_target(clean)

    _run_session_panel(sess, distance_m, clicks_per_cm)


if __name__ == "__main__":
    run_interactive_session()


[session] distance=50 m | 2 clicks/cm -> 0.5000 cm/click
[session] baseline set - center template 144x144px at (450,600)

[session] processing round #1
[session] group center=(490.1,646.5); windage +2.0 clicks, elevation +2.3 clicks

[session] processing round #2
[session] group center=(350.0,520.0); windage -5.0 clicks, elevation -4.0 clicks
[session] ended after 2 round(s).
